# 06 - M6: steering gates (real CLIP, real T4)
**Runtime > T4 GPU.** Gate: with gates at init, steered output equals plain output exactly; gates receive gradient.
Also checked on the real pipeline: steering path alone reaches the front end and LoRA, causality survives steering, fp16 is finite, and memory/speed vs the plain stack (budget: peak < 10 GiB at batch 8).

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'master'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import json, time, os, numpy as np, torch
from datasets import load_dataset
from transformers import CLIPVisionModel, CLIPTextModelWithProjection, CLIPTokenizer
from src.data import load_manifest, ManifestDataset
from src.backbone import ClipLoRABackbone
from src import concepts as K
from src.fusion import ConceptFrontEnd
from src.mamba import PlainMamba
from src.steering import SteeredMamba, gate_stats
dev = 'cuda'
res = {'milestone': 'M6'}

ASSET = 'assets/concepts_text_768.npy'          # committed after M4; if absent, encode it now (same code as notebook 04)
if os.path.exists(ASSET):
    T = K.load_or_encode(ASSET)
else:
    tm = CLIPTextModelWithProjection.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32)
    T = K.load_or_encode(ASSET, tm, CLIPTokenizer.from_pretrained(K.TEXT_CHECKPOINT), device='cuda'); del tm
mf = load_manifest('Midjourney')
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=mf['revision'])
tr = ManifestDataset(mf, 'train', ds['train'], ds['validation'], protocol='controlled', train=False)
xb = torch.stack([tr[i][0] for i in range(8)]).to(dev)
clip = CLIPVisionModel.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32).to(dev).eval()
bb = ClipLoRABackbone(clip, taps=(16, 24), r=8, alpha=16.0).to(dev)
fe = ConceptFrontEnd(T, d_in=1024, d=384, n_taps=2).to(dev)

torch.manual_seed(0)
plain = PlainMamba(2, chunk=8).to(dev).eval()          # chunk 8 was ~40% faster than 16 at equal memory in M5
steered = SteeredMamba.from_plain(plain, ).to(dev).eval()   # same mixer weights, gates at identity
print('mixer params', sum(p.numel() for p in steered.mixer_parameters()), '| gate params', sum(p.numel() for p in steered.gate_parameters()))
assert sum(p.numel() for p in steered.gate_parameters()) == 27_200

def feats(autocast):
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=autocast):
        o = fe(bb(xb))
    return o['X'], o['alpha']

In [ ]:
# GATE 1: with gates at init, steered output == plain output EXACTLY, on the real pipeline (real CLIP features, real alpha)
g1 = {}
for mode in ('fp32', 'fp16'):
    X, alpha = feats(mode == 'fp16')
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16, enabled=(mode == 'fp16')):
        p1, p2, s = plain(X), plain(X), steered(X, alpha)
    base = (p1.float() - p2.float()).abs().max().item()      # run-to-run noise of the plain stack itself (expected 0.0)
    d = (p1.float() - s.float()).abs().max().item()
    g1[mode] = {'plain_vs_plain': base, 'steered_vs_plain': d, 'exact': bool(torch.equal(p1, s))}
    print(mode, g1[mode])
    assert d <= base, f'steered differs from plain at init ({mode}): {d} > baseline {base}'
ac = (16 * alpha - 1).abs()
res['gate1_init_equals_plain'] = g1
res['gate_input_stats_init'] = {'mean_abs_16a_minus_1': ac.mean().item(), 'max_abs_16a_minus_1': ac.max().item()}
print('input to the gates, |16*alpha-1|:', res['gate_input_stats_init'])
print('GATE 1 OK')

In [ ]:
# GATE 2: gates receive gradient. (a) at init, on the real pipeline: every steered-stack parameter gets gradient.
# (b) once the gates are non-zero, the front end and LoRA get gradient THROUGH THE STEERING PATH ONLY (X is detached
# before the Mamba, so the only way a gradient can reach fe/LoRA is alpha -> gates). Loss = sum of squares (not mean):
# under fp16 autocast a tiny mean-loss gradient can underflow and fake a 'dead' parameter.
g = lambda p: p.grad is not None and p.grad.abs().sum().item() > 0
st = SteeredMamba.from_plain(plain).to(dev).train()
for mod in (bb, fe, st): mod.zero_grad(set_to_none=True)
with torch.autocast('cuda', dtype=torch.float16):
    o = fe(bb(xb)); H = st(o['X'], o['alpha'])
H.float().square().sum().backward()
dead = [n for n, p in st.named_parameters() if not g(p)]
assert not dead, dead
gate_grad = {n: p.grad.abs().sum().item() for n, p in st.named_parameters() if '.gates.' in n}
print('(a) all', len(list(st.parameters())), 'steered-stack params have gradient at init; gate grads:', gate_grad)

with torch.no_grad():
    for p in st.gate_parameters(): p.normal_(0, 0.1)
    for n, p in bb.named_parameters():
        if 'lora_B' in n: p.add_(0.01 * torch.randn_like(p))      # B=0 at init zeroes A's gradient
for mod in (bb, fe, st): mod.zero_grad(set_to_none=True)
with torch.autocast('cuda', dtype=torch.float16):
    o = fe(bb(xb)); H = st(o['X'].detach(), o['alpha'])
LOSS_SCALE = 1024.0   # what GradScaler does in M8: the steering-path gradient reaching LoRA is tiny and underflows in fp16 otherwise
(H.float().square().sum() * LOSS_SCALE).backward()
assert g(fe.affinity.text_proj.weight), 'text projection got no gradient via steering'
assert g(fe.fusion.proj.weight), 'visual projection got no gradient via steering'
assert g(fe.affinity.log_scale) and g(fe.fusion.mix_logits)
assert fe.affinity.text_embeds.grad is None
lp = dict(bb.named_parameters())
dead = [n for n, p in lp.items() if 'lora_' in n and not g(p)]
bad = [n for n, p in lp.items() if 'lora_' in n and p.grad is not None and not torch.isfinite(p.grad).all()]
print('LoRA tensors:', sum('lora_' in n for n in lp), '| zero-grad:', len(dead), dead[:6], '| non-finite:', len(bad), bad[:3])
assert not dead and not bad, 'LoRA got no (or non-finite) gradient via steering'
res['gate2_gradients'] = {'init_all_params_have_grad': True, 'gate_grad_abs_sums_init': gate_grad, 'steering_path_reaches_frontend_and_lora': True}
print('(b) steering path alone reaches text_proj, fusion.proj, log_scale, mix_logits and all LoRA tensors')
print('GATE 2 OK | gate stats', gate_stats(st)[0])

In [ ]:
# GATE 3: steering is live and stable on real data (gates nudged non-zero): output differs from plain, depends on alpha,
# finite in fp16, and the causality gate from M5 still holds (perturb token t AND its alpha; nothing before t may move)
X, alpha = feats(True)
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    a = st.eval()(X, alpha)
    pl = plain(X)
    a_shuf = st(X, alpha.roll(1, 0))
live = {'vs_plain': (a.float() - pl.float()).abs().max().item(), 'vs_other_alpha': (a.float() - a_shuf.float()).abs().max().item()}
print(live)
assert torch.isfinite(a).all() and live['vs_plain'] > 1e-3 and live['vs_other_alpha'] > 1e-3
worst = {}
for t in (0, 1, 15, 16, 287, 575):
    X2, al2 = X.clone(), alpha.clone()
    X2[:, t] += 5.0; al2[:, t] = alpha.flip(0)[:, t]
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        o1, o2 = st(X, alpha), st(X2, al2)
    before = (o1[:, :t].float() - o2[:, :t].float()).abs().max().item() if t else 0.0
    after = (o1[:, t:].float() - o2[:, t:].float()).abs().max().item()
    worst[t] = (before, after)
    assert before < 1e-4, (t, before)
    assert after > 1e-3, (t, after)
print({k: (f'{b:.1e}', f'{a_:.1e}') for k, (b, a_) in worst.items()})
res['gate3_live_and_causal'] = {'live': live, 'causality_before_after': {str(k): v for k, v in worst.items()}}
print('GATE 3 OK')

In [ ]:
# GATE 4: memory + speed, full pipeline (CLIP+LoRA -> front end -> 2 Mamba blocks), B=8, fp16 autocast, fwd+bwd.
# Same budget as M5 (D008): peak < 10 GiB. M5 plain baselines: chunk 16 = 8.47 GiB / 7.6 img/s.
def run(kind, chunk, steps=3):
    torch.manual_seed(0)
    mm = (PlainMamba(2, chunk=chunk) if kind == 'plain' else SteeredMamba(2, chunk=chunk)).to(dev)
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    ts = []
    for _ in range(steps):
        for mod in (bb, fe, mm): mod.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); t0 = time.time()
        with torch.autocast('cuda', dtype=torch.float16):
            o = fe(bb(xb))
            H = mm(o['X']) if kind == 'plain' else mm(o['X'], o['alpha'])
        H.float().square().mean().backward()
        torch.cuda.synchronize(); ts.append(time.time() - t0)
    assert torch.isfinite(H).all()
    sec = float(np.median(ts))
    return {'peak_gib': torch.cuda.max_memory_allocated() / 2**30, 'sec_per_step_b8': sec, 'img_per_s': 8 / sec,
            'est_min_per_epoch_3240_imgs': 3240 / (8 / sec) / 60}
grid = {}
for kind, chunk in (('plain', 8), ('steered', 8), ('steered', 16)):
    grid[f'{kind}_chunk{chunk}'] = run(kind, chunk); print(kind, chunk, grid[f'{kind}_chunk{chunk}'])
res['pipeline_b8'] = grid
assert grid['steered_chunk8']['peak_gib'] < 10.0, 'steered default over budget: use checkpoint=True (see M5 grid)'
print('GATE 4 OK | steered chunk 8 peak', f"{grid['steered_chunk8']['peak_gib']:.2f} GiB",
      '| overhead vs plain', f"{grid['steered_chunk8']['peak_gib'] - grid['plain_chunk8']['peak_gib']:+.2f} GiB,",
      f"{grid['steered_chunk8']['sec_per_step_b8'] / grid['plain_chunk8']['sec_per_step_b8'] - 1:+.1%} time")

In [ ]:
res['gate'] = 'PASS'
for p in (f'{DRIVE_ROOT}/runs/m6_gate.json', '/content/csm-clip/runs/m6_gate.json'):
    json.dump(res, open(p, 'w'), indent=1)
print(json.dumps(res, indent=1))
# Then: commit runs/m6_gate.json, paste docs/d007_row.md over the D007 row in docs/decision_log.md,
# set M6 to Done in docs/contract.md, tag m6-done.